# pctrl：最小化 HV / VH / DA / AD 四组双光子计数

这个示例复用 `qtools.pctrl.optimize.BasisMeasurement` 和唯一的 `pctrl.optimize()` 优化入口。

电机连接参数从 `motor_config.yaml` 读取，basis 校准角度从 `polmeas_config.yaml` 读取。


## 1. 配置

请在实验目录中启动 Jupyter，并把 TDC 的窗口参数改成当前实验值。`TARGET` 可以是一个 channel（单侧优化），也可以是 `["alice", "bob"]`（双侧 joint 优化）。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from qtools.motor import MotorConfigManager
from qtools.pctrl import BasisMeasurement, PolarizationController
from qtools.pmeas.config import ConfigManager as PolmeasConfigManager
from qtools.pmeas.motor_manager import MotorManager

PCTRL_CONFIG = Path.cwd() / "pctrl_config.yaml"
MOTOR_CONFIG = Path.cwd() / "motor_config.yaml"
POLMEAS_CONFIG = Path.cwd() / "polmeas_config.yaml"
TARGET = ["alice", "bob"]  # use "alice" for one-sided optimization

MINIMIZATION_BASIS = [("H", "V"), ("V", "H"), ("D", "A"), ("A", "D")]
MAXIMIZATION_BASIS = []

MEASURE_DURATION = 1.0
MEASURE_SAMPLES = 3
CH_START, CH_STOP = 1, 4
DELAY = 100
WINDOW_START, WINDOW_STOP = 109, 110
THRESHVOLT = 0.6

MAXITER = 40
ANGLE_TOL = 1.0
STEP_SIZE_DEG = 15.0
SPEED_DEG_S = 10.0
SETTLE_TIME = 0.2
OVERSHOOT = True  # enable the two-step move for servo displacements < 3 degrees

for path in (PCTRL_CONFIG, MOTOR_CONFIG, POLMEAS_CONFIG):
    print(f"{path}: {'OK' if path.exists() else 'missing'}")


## 2. 创建控制器和 basis measurement

`BasisMeasurement` 负责切换 Alice/Bob basis 和读取 TDC；`pctrl.optimize()` 负责唯一的波片优化循环。loss 自动由 basis 列表组成：

`loss = sum(rate**2 for rate in MINIMIZATION_BASIS) - sum(rate**2 for rate in MAXIMIZATION_BASIS)`


In [ ]:
pctrl = PolarizationController(config_path=str(PCTRL_CONFIG))
motor_cfg = MotorConfigManager.resolve(MOTOR_CONFIG)
polmeas_cfg = PolmeasConfigManager.resolve(POLMEAS_CONFIG)
motor_mgr = MotorManager(motor_cfg)

basis_measurement = BasisMeasurement(
    meas_mgr=pctrl.meas_mgr,
    polmeas_cfg=polmeas_cfg,
    motor_mgr=motor_mgr,
    minimization_basis=MINIMIZATION_BASIS,
    maximization_basis=MAXIMIZATION_BASIS,
)
print("controllers ready")


## 3. 运行唯一的 pctrl.optimize

每次评估会自动完成四次 pairs 测量，并最小化四组 rate 的平方和。若需要在某些 basis 上最大化，把它们放进 `MAXIMIZATION_BASIS`；它们会以负号进入 loss。

本例设置 `OVERSHOOT = True`：当 pctrl 舵机目标角度与实际编码器角度的非零差值小于 3° 时，先转动一个大角度，再回到目标角度，等待到位后测量。差值达到或超过 3° 时直接移动。设为 `False` 可关闭；API 默认关闭。


In [ ]:
def progress(iteration, angles, value, _raw):
    basis_measurement.history_rows.append((
        iteration, dict(basis_measurement.last_counts), float(value)
    ))
    counts = " ".join(
        f"{a}{b.lower()}={basis_measurement.last_counts[(a, b)]:.2f}"
        for a, b in basis_measurement.measurement_keys
    )
    print(f"eval {iteration:03d} | loss={value:.3f} | {counts}")

result = pctrl.optimize(
    target=TARGET,
    measurement_adapter=basis_measurement,
    minimize=True,
    method="Nelder-Mead",
    maxiter=MAXITER,
    tol=ANGLE_TOL,
    step_size_deg=STEP_SIZE_DEG,
    speed_deg_s=SPEED_DEG_S,
    settle_time=SETTLE_TIME,
    overshoot=OVERSHOOT,
    duration=MEASURE_DURATION,
    samples=MEASURE_SAMPLES,
    measure_type="pairs",
    ch_start=CH_START,
    ch_stop=CH_STOP,
    delay=DELAY,
    window_start=WINDOW_START,
    window_stop=WINDOW_STOP,
    threshvolt=THRESHVOLT,
    callback=progress,
    verbose=True,
)
print(result.message)
print("optimal angles:", result.optimal_angles)


## 4. 查看四组计数和收敛过程

历史记录由 `BasisMeasurement` 保存。


In [ ]:
rows = basis_measurement.history_rows
labels = ["".join(key) for key in basis_measurement.measurement_keys]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([row[0] for row in rows], [row[2] for row in rows], marker="o")
axes[0].set(xlabel="evaluation", ylabel="loss", title="basis loss")
axes[0].grid(True, alpha=0.3)

for index, label in enumerate(labels):
    key = basis_measurement.measurement_keys[index]
    axes[1].plot(
        [row[0] for row in rows],
        [row[1][key] for row in rows],
        marker="o",
        label=label,
    )
axes[1].set(xlabel="evaluation", ylabel="coincidence rate (cps)", title="basis rates")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


## 5. 关闭连接

完成实验后执行。

In [ ]:
try:
    pctrl.close()
finally:
    motor_mgr.disconnect_all()
print("hardware connections closed")
